# Plot players seasons

In [ ]:
from datetime import datetime

import matplotlib.pyplot as plt
import seaborn as sns

from meml_griz import fetch

Get 2025-2026 season

In [ ]:
from nba_api.stats.library.http import NBAStatsHTTP

session = cr.Session(impersonate="chrome120")
session.get("https://www.nba.com/stats/", timeout=30)  # warmup
NBAStatsHTTP.get_session = lambda self: session

# Download all team games since 2022-2023 season
for team in (fetch.NbaTeams.MEM,):
    team_seasons = fetch.SeasonData.from_date_range(
        team=team,
        start_date=datetime(2020, 10, 1),
        end_date=datetime(2026 + 1, 6, 1),
    )

Plot players PPG

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(8, 8), sharex=True)


# Get active games
active_games = team_seasons.season_df[
    (team_seasons.season_df["isActive"])
    & (team_seasons.season_df.gameParticipationRate > 25)
].copy()
active_games["medianNetRating"] = active_games.groupby("playerSlug")[
    "netRating"
].transform("median")
active_games["meanNetRating"] = active_games.groupby("playerSlug")[
    "netRating"
].transform("mean")

# Calculate season metrics for each player
df = active_games.groupby("playerSlug", as_index=False).agg(
    meanNetRating=("netRating", "mean"),
    stdNetRating=("netRating", "std"),
    medianNetRating=("netRating", "median"),
)
# Sort dataframe
active_games = active_games.sort_values(by=["meanNetRating"], ignore_index=True)

# Plot sub-plots
ax = axes[0]
sns.lineplot(
    data=active_games,
    x="playerSlug",
    y="netRating",
    errorbar="sd",
    err_style="bars",
    err_kws={"capsize": 5},
    marker="s",
    markeredgecolor="none",
    sort=False,
    ax=ax,
)
ax.grid()
ax.tick_params(rotation=90)


for ycol, ax in zip(["stdNetRating", "medianNetRating"], axes[1:].ravel()):
    sns.barplot(
        data=df,
        x="playerSlug",
        y=ycol,
        ax=ax,
    )
    ax.set_axisbelow(True)
    ax.grid(axis="y")
    ax.tick_params(rotation=90)

plt.show()

## Plot player stats over time

In [ ]:
import pandas as pd

# Get only active games
active_games = team_seasons.season_df[
    (team_seasons.season_df["isActive"])
    & (team_seasons.season_df["gameParticipationRate"] > 0.25)
].copy()

# Get top players by netRating
top_players = (
    active_games.groupby(["playerSlug"])["minutes"].mean().nlargest(15)
).index
active_games = active_games[active_games["playerSlug"].isin(top_players)]

active_games["gameDate"] = pd.to_datetime(active_games["gameDate"])
active_games["month"] = active_games["gameDate"].dt.month
active_games["year"] = active_games["gameDate"].dt.year
active_games["day"] = active_games["gameDate"].dt.day

# Group by year, month, player. Get the stat per month
df = (
    active_games.groupby(["year", "month", "playerSlug"])
    .agg(
        meanNetRating=("netRating", "mean"),
        stdNetRating=("netRating", "std"),
        medianNetRating=("netRating", "median"),
        meanMinutes=("minutes", "mean"),
    )
    .reset_index()
)

# Get x-axis for sns plot
df["year_month"] = pd.to_datetime(df[["year", "month"]].assign(day=1))

Plot the stats over time

In [ ]:
fig, axes = plt.subplots(1, 2, constrained_layout=True, figsize=(8, 6))

for ycol, ax in zip(["meanNetRating", "medianNetRating"], axes.ravel()):
    sns.lineplot(
        data=df,
        x="year_month",
        y=ycol,
        ax=ax,
        hue="playerSlug",
        palette="tab20",
        linewidth=1.5,
    )
    ax.set_axisbelow(True)
    ax.grid(axis="y")
    ax.tick_params(rotation=90)

    if ax != axes[0]:
        ax.legend().set_visible(False)